# 01_exploration_openchargemap.ipynb

In [13]:
from dotenv import load_dotenv
import os
import requests
from datetime import datetime
import json
from pathlib import Path

ROOT_PATH = Path.cwd().resolve().parent

load_dotenv(dotenv_path="../.env")  # cherche un fichier .env dans le dossier courant (ou parent)
cle_api = os.environ.get("OCM_API_KEY")


url = "https://api.openchargemap.io/v3/poi"

querystring = {
                "output":"json",
                "key":f"{cle_api}",
                "latitude" : 48.856614,
                "longitude" : 2.3522219,
                "distance" : 5,
                "distanceunit" : "km",
                "maxresults" : 50
            }
     
headers = {
            "Accept": "application/json",
             "User-Agent": "electric-mobility-platform/0.1 (projet portfolio Data Engineering)"}

response = requests.get(url, headers=headers, params=querystring)
print(response.status_code)

if response.status_code == 200:   
    now = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    path_target_file = ROOT_PATH / "data" / "raw" / f"{now}_paris_extract.json"
    
    with open(path_target_file, "w") as f:
        json.dump(response.json(), f)
else :
    print(f"Échec de la requête, code {response.status_code}, détail {response.text}")

200


## Upload vers AWS S3

In [14]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError
import os

# Création du client AWS
s3_client = boto3.client(
    "s3",
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY"),
    region_name=os.environ.get("AWS_REGION")
)

# Fichier à uploader sur S3
target_file = "2026-07-24_103154_paris_extract.json"
target_path = ROOT_PATH / "data" / "raw" / target_file

# Nom du bucket cible
target_s3_bucket_name = "electric-mobility-platform-thierry"

# Destination cible dans le bucket s3
target_s3_destination = "raw/" + target_file

# Upload du fichier sur S3
try:
    s3_client.upload_file(str(target_path), target_s3_bucket_name, target_s3_destination)
    print(f"Upload réussi : {target_s3_destination}")
except NoCredentialsError:
    print("Erreur : identifiants AWS manquants ou invalides.")
except ClientError as e:
    print(f"Erreur AWS lors de l'upload : {e}")

Upload réussi : raw/2026-07-24_103154_paris_extract.json


## Questions exploration

Est-ce que Connections est vide sur toutes les entrées, ou seulement certaines ?
Quels champs sont toujours présents vs parfois None ? (OperatorInfo, UsageCost, GeneralComments sont déjà None sur tes deux résultats)
Quelle est la structure d'un élément de Connections quand il n'est pas vide ? (regarde la doc de référence /v3/referencedata/ si tu veux comprendre les codes utilisés dedans, comme les types de connecteurs)

## Est-ce que Connections est vide sur toutes les entrées, ou seulement certaines ?

In [15]:
resultats = response.json()

#Est-ce que Connections est vide sur toutes les entrées, ou seulement certaines ?

# Liste des connexions
connections_list = [poi["Connections"] for poi in resultats]

# Nombre de connexions vides
nb_empty_connections = len([connection for connection in connections_list if len(connection) == 0])

# Pourcentage de connexions vides
percentage_empty_connections = nb_empty_connections/len(connections_list)*100

print(f"Dans cet extract, {percentage_empty_connections} % des connections sont vides ! ")

Dans cet extract, 30.0 % des connections sont vides ! 


## Quels champs sont toujours présents vs parfois None ?

In [16]:
# Quels champs sont toujours présents vs parfois None ?

#est-ce que toutes mes entrées ont les mêmes clés, oui ou non

#Liste des clés de chaque poi (utilisation de frozenset)
frozen_keys = [frozenset(poi.keys()) for poi in resultats]

# est-ce que toutes mes entrées ont les mêmes clés, oui ou non ?
freeze_all = set(frozen_keys)
print(len(freeze_all))

1


In [17]:
# pour les clés communes, regarde combien ont une valeur None vs une vraie valeur.
keys = resultats[0].keys()

from collections import Counter

cnt_counter = Counter()

for poi in resultats:
    for key in keys:
        if poi[key] is None:
            cnt_counter[key] += 1
display(cnt_counter)

# Cas spécifique : Connections est une liste, pas juste None/valeur
connections_vides = sum(1 for poi in resultats if not poi["Connections"])
print(connections_vides)

Counter({'UserComments': 50,
         'PercentageSimilarity': 50,
         'MediaItems': 50,
         'ParentChargePointID': 50,
         'DatePlanned': 50,
         'MetadataValues': 50,
         'OperatorsReference': 43,
         'OperatorInfo': 41,
         'OperatorID': 41,
         'DateLastConfirmed': 35,
         'UsageCost': 29,
         'NumberOfPoints': 28,
         'DataProvidersReference': 23,
         'GeneralComments': 6})

15
